# Data Preprocessing

Flatten collected YouTube comments and replies into clean rows for downstream analysis.


In [2]:
import json
from pathlib import Path
import pandas as pd
import random 
from langdetect import detect, LangDetectException
from collections import Counter
import nltk
import string
import re
import html
import unicodedata
import emoji
from nltk.corpus import stopwords


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
VIDEO_DATA_PATH = DATA_DIR / "video_data.json"
PROCESSED_VIDEO_DATA_PATH = DATA_DIR / "video_data_processed.json"
MET_GALA_ENTITIES_PATH = DATA_DIR / "met_gala_entities.json"

RANDOM_SEED = 42

nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)


True

In [3]:
with open(VIDEO_DATA_PATH, "r", encoding="utf-8") as f:
    video_data = json.load(f)["videos"]

print("Video data loaded")
print("Total videos:", len(video_data))
print("Total collected comment rows:", sum(len(video.get("comments", [])) for video in video_data))


Video data loaded
Total videos: 110
Total collected comment rows: 63250


In [4]:
comments_flattened = []
seen_comment_ids = set()
duplicate_comment_rows = 0
for video in video_data:
    video_context = {
        "video_id": video.get("videoId"),
        "video_title": video.get("title"),
        "channel_id": video.get("channelId"),
        "channel_title": video.get("channelTitle"),
        "video_published_at": video.get("publishedAt"),
        "video_view_count": video.get("viewCount", 0),
        "video_like_count": video.get("likeCount", 0),
        "video_available_comment_count": video.get("commentCount", 0),
    }

    for comment in video.get("comments", []):
        comment_id = comment.get("commentId")
        if comment_id in seen_comment_ids:
            duplicate_comment_rows += 1
            continue
        seen_comment_ids.add(comment_id)

        comments_flattened.append({
            **video_context,
            "comment_id": comment_id,
            "comment_text": comment.get("text", ""),
            "comment_author_id": comment.get("authorId"),
            "comment_author": comment.get("author"),
            "comment_published_at": comment.get("publishedAt"),
            "comment_updated_at": comment.get("updatedAt"),
            "comment_like_count": comment.get("likeCount", 0),
            "is_reply": comment.get("isReply", False),
            "parent_comment_id": comment.get("parentCommentId"),
            "reply_to_author_id": comment.get("replyToAuthorId"),
            "top_level_reply_count": comment.get("totalReplyCount", 0),
            "text_length": len(comment.get("text", "") or ""),
        })


In [5]:

total_comments = len(comments_flattened)
total_replies = sum(1 for c in comments_flattened if c.get("is_reply") == True)
total_parent_comments = total_comments - total_replies
total_empty_comments = sum(1 for c in comments_flattened if not c.get("comment_text", "").strip())

print("COMMENT FLATTENING SUMMARY")
print("=" * 80)
print(f"Flattened comment rows: {total_comments}")
print(f"Duplicate comment rows removed: {duplicate_comment_rows}\n")

print(f"Total comments: {total_comments}")
print(f"Total parent comments: {total_parent_comments}")
print(f"Total replies: {total_replies}")
print(f"Total empty comments: {total_empty_comments}")

COMMENT FLATTENING SUMMARY
Flattened comment rows: 63048
Duplicate comment rows removed: 202

Total comments: 63048
Total parent comments: 48950
Total replies: 14098
Total empty comments: 13


In [6]:

TOTAL_RANDOM_SAMPLES = 25

print("\nRANDOM SAMPLE OF COMMENTS TO IDENTIFY ISSUES")
print("=" * 80)
random.seed(RANDOM_SEED)
random_sample_indices = random.sample(range(len(comments_flattened)), min(TOTAL_RANDOM_SAMPLES, len(comments_flattened)))
for i, idx in enumerate(random_sample_indices):
    text = comments_flattened[idx]['comment_text'].strip().replace('\n', ' ').replace('\r', '')
    text = ' '.join(text.split())
    print(f"[{i+1}/{TOTAL_RANDOM_SAMPLES}] {text[:150]:<10}")



RANDOM SAMPLE OF COMMENTS TO IDENTIFY ISSUES
[1/25] there is the stereotype and then there is bad. u won the award for both. just to show u just cause u got monie, doesnt mean shite. especially abiut fa
[2/25] 14:30 in my opinion, it needed a big sumptuous cloak, maybe with a hood and gloves, to be more evocative of how luscious and involved klimt’s pieces a
[3/25] what is cara doing :/
[4/25] I’m actually crying laughing on my way to work! Love you Zach! Can’t wait to see you hosting one day ❤
[5/25] Nicole Kidman is always pure class. She is stunning.
[6/25] Megyn is sooo jealous she wasn't invited
[7/25] My only look at the met gala, thanks Garrron! Getting Halloween in springtime vibes 🤡👻👽👀
[8/25] I agree with so many of your critiques. The fact Anna didn't even bother with the theme and wore a variation of a previous dress let's me know how fri
[9/25] ​@prpowell4038😢 for the fact she's entitled she might be acting the same way as North. Get out of gay Zee's and be-yawn🥱nce's asse

In [7]:
def detect_language(text):
    """Detect language, returning ISO code."""
    try:
        if not text or not len(text.strip()):
            return 'en'
        return detect(text)
    except LangDetectException:
        return 'unknown'

# Collect comment rows from the rows list
language_results = [(comment, detect_language(comment.get('comment_text', ''))) for comment in comments_flattened]
language_counter = Counter(lang for _, lang in language_results)

In [8]:
TOP_LANGUAGE_COUNT = 10
TOTAL_NON_ENGLISH_EXAMPLES = 20

total_comments = len(language_results)

print(f"\nTOP {TOP_LANGUAGE_COUNT} LANGUAGE DETECTIONS")
print("=" * 80)
for i, (lang, count) in enumerate(language_counter.most_common(TOP_LANGUAGE_COUNT), start=1):
    pct = 100 * count / total_comments
    print(f"[{i}] {lang} {count:,} ({pct:.2f}%)")
    if i == 10:
        break


TOP 10 LANGUAGE DETECTIONS
[1] en 46,808 (74.24%)
[2] unknown 1,679 (2.66%)
[3] so 1,541 (2.44%)
[4] pt 1,153 (1.83%)
[5] de 1,058 (1.68%)
[6] tl 1,046 (1.66%)
[7] af 962 (1.53%)
[8] fr 833 (1.32%)
[9] et 810 (1.28%)
[10] id 771 (1.22%)


In [9]:
COMMENT_TRUNCATION_LENGTH = 200
ENGLISH_FILTER = "en"

# Extract full comment data for English comments
english_comments = [comment for comment, lang in language_results if lang == ENGLISH_FILTER]
for comment in english_comments:
    comment["language"] = ENGLISH_FILTER

# Overwrite comments flattened with English filtered list
comments_flattened = english_comments

# Extract truncated comment data for non-English comments for test display
non_english_comments = [comment['comment_text'][:COMMENT_TRUNCATION_LENGTH] for comment, lang in language_results if lang != 'en']

random.seed(RANDOM_SEED)
random_non_english = random.sample(non_english_comments, min(TOTAL_NON_ENGLISH_EXAMPLES, len(non_english_comments)))
random_english = random.sample(comments_flattened, min(TOTAL_NON_ENGLISH_EXAMPLES, len(comments_flattened)))

In [10]:

print("\nRANDOM NON-ENGLISH COMMENTS REMOVED:")
print("=" * 80)
for idx, text in enumerate(random_non_english):
    print(f"[{idx+1}/{TOTAL_NON_ENGLISH_EXAMPLES}] {text}")



RANDOM NON-ENGLISH COMMENTS REMOVED:
[1/20] mango from jamnagar
[2/20] Jisooo 😍😍
[3/20] I love lisa's look
[4/20] Ralph lauren ❤
[5/20] Лlol😅
[6/20] Drawer under your TV.😂
[7/20] RACHEL ZEGLAZOID!
[8/20] LISA 🪽🪽🪽🪽
[9/20] Se ve tan sencillo y agradable
[10/20] Vine por las Pinks. Que emocionnnn❤❤❤❤❤
[11/20] ❤❤❤❤❤❤❤
[12/20] 🥰❤️👏🏾
[13/20] Kim ❤❤❤❤❤
[14/20] not harden scott😭😭
[15/20] Met Gala ❌ pinkGala ✅
[16/20] Georgina
[17/20] Why?
[18/20] Hecham chiroyli emas
[19/20] Margot Robbie
[20/20] Hunger games


In [11]:
print("\nRANDOM ENGLISH COMMENTS REMAINING:")
print("=" * 80)
for idx, comment in enumerate(random_english):
    print(f"[{idx+1}/{TOTAL_NON_ENGLISH_EXAMPLES}] {comment['comment_text'][:COMMENT_TRUNCATION_LENGTH]}")



RANDOM ENGLISH COMMENTS REMAINING:
[1/20] Russell and Ciara shut it down!!
[2/20] Marie Antoinette???? Let them eat cake !!!! this met gala is absurd !!!!!!
[3/20] Bullfighting theme? 🥺☹️
[4/20] What was the theme?
[5/20] Where is emma
[6/20] Let’s not forget how Heidi is all over the Epstein files and was besties with gelane
[7/20] Anna Wintour is British and they have been actively trying to destroy our culture.
[8/20] Demons rather crawl as you know
[9/20] Rosé is the best in the world❤she is pretty not ugly like in the coments
[10/20] Beyonce encouraging too 🤣🤣 love this for Blue
[11/20] Lizzo was a hot bug indeed
[12/20] ....what a ghastly affair.
[13/20] 5:18 its kinda giving the drug ballons from euphoria 😂
[14/20] Almost nobody’s stylist understood the theme
[15/20] Lisa and jisoo killed this shit .... I love their looks so much ❤❤❤❤❤
[16/20] The hunger games capital residents are running wild
[17/20] Best review by far an absolutely delicious treat
[18/20] The worst dressed w

In [12]:
english_count = len(english_comments)
removed_count = total_comments - english_count
english_pct = 100 * english_count / total_comments
removed_pct = 100 * removed_count / total_comments

print("ENGLISH FILTERING SUMMARY")
print(80 * "=")
print(f"\nEnglish kept: {english_count} ({english_pct:.1f}%)")
print(f"Non-English removed: {removed_count} ({removed_pct:.1f}%)")

ENGLISH FILTERING SUMMARY

English kept: 46808 (74.2%)
Non-English removed: 16240 (25.8%)


In [13]:
# Preprocessing constants and helper functions
URL_PATTERN = re.compile(r'https?://\S+')
TIMESTAMP_PATTERN = re.compile(r'\b\d{1,2}:\d{2}(?::\d{2})?\b')
MENTION_PATTERN = re.compile(r'@[\w.-]+[\w]')
DIGIT_PATTERN = re.compile(r'\d+')
PUNCT_PATTERN = re.compile(r'[^\w\s]')
ENTITY_SEPARATOR_PATTERN = re.compile(r'[^a-z\s]+')

TWEET_TOKENISER = nltk.tokenize.TweetTokenizer(
    reduce_len=True,
    strip_handles=True,
    preserve_case=False,
)

PUNCTUATION = list(string.punctuation)
TWEET_STEMMER = nltk.stem.PorterStemmer()
STOP_WORDS = set(stopwords.words('english')) | set(PUNCTUATION)

def remove_html_entities(text):
    return html.unescape(text or "")

def remove_urls(text):
    return URL_PATTERN.sub("", text)

def remove_timestamps(text):
    return TIMESTAMP_PATTERN.sub("", text)

def remove_mentions(text):
    return MENTION_PATTERN.sub("", text)

def remove_accents(text):
    return unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode()

def remove_unicode(text):
    return text.encode("ascii", "ignore").decode()

def remove_digits(text):
    return DIGIT_PATTERN.sub(" ", text)

def strip_punctuation(text):
    return PUNCT_PATTERN.sub(" ", text)

def strip_entity_separators(text):
    return ENTITY_SEPARATOR_PATTERN.sub(" ", text)

def remove_emojis(text):
    return emoji.replace_emoji(text, replace="")

def normalise_whitespace(text):
    return " ".join(text.split())

def tokenize(text):
    return TWEET_TOKENISER.tokenize(text)

def remove_stopwords(tokens):
    return [token for token in tokens if token not in STOP_WORDS]

def stem_tokens(tokens):
    return [TWEET_STEMMER.stem(token) for token in tokens]

def clean_common_text(text):
    text = remove_html_entities(text)
    text = remove_urls(text)
    text = remove_timestamps(text)
    return remove_mentions(text)

def clean_for_entity_matching(text):
    text = clean_common_text(text)
    text = remove_accents(text.lower())
    text = strip_entity_separators(text)
    return normalise_whitespace(text)

def sentiment_analysis_clean(text):
    """Minimal cleaning appropriate for VADER and BERT sentiment analysis."""
    return normalise_whitespace(clean_common_text(text))

def clean_for_topic_text(text):
    text = clean_for_entity_matching(text)
    text = remove_unicode(text)
    text = remove_digits(text)
    text = strip_punctuation(text)
    return normalise_whitespace(text)

def clean_for_topic_tokens(text):
    text = clean_for_topic_text(text)
    tokens = tokenize(text)
    return remove_stopwords(tokens)

def clean_for_topic_stemmed_tokens(text):
    return stem_tokens(clean_for_topic_tokens(text))

def purify_text(text, show_changes=False):
    """Return heavily cleaned, tokenized text with stopwords removed."""
    if not show_changes:
        return clean_for_topic_tokens(text)

    history = {}
    text = remove_html_entities(text)
    history["remove_html_entities"] = text
    text = remove_urls(text)
    history["remove_urls"] = text
    text = remove_timestamps(text)
    history["remove_timestamps"] = text
    text = remove_mentions(text)
    history["remove_mentions"] = text
    text = text.lower().strip()
    history["lowercase_and_strip"] = text
    text = normalise_whitespace(text)
    history["normalise_whitespace"] = text
    text = remove_unicode(text)
    history["remove_unicode"] = text
    text = remove_digits(text)
    history["remove_digits"] = text
    text = strip_punctuation(text)
    history["strip_punctuation"] = text
    text = normalise_whitespace(text)
    history["normalise_whitespace_after_punctuation"] = text
    tokens = tokenize(text)
    history["tokenize"] = tokens
    tokens = remove_stopwords(tokens)
    history["remove_stopwords"] = tokens
    history["stem_tokens"] = stem_tokens(tokens)
    return history


In [14]:
# Store lightly cleaned text for exact entity matching
for comment in comments_flattened:
    comment["comment_text_entity"] = clean_for_entity_matching(comment.get("comment_text", ""))

print("Lightly cleaned text ready for entity matching")


Lightly cleaned text ready for entity matching


In [15]:
unique_videos = set(comment['video_id'] for comment in comments_flattened)
unique_channels = set(comment['channel_id'] for comment in comments_flattened)
unique_authors = set(comment['comment_author'] for comment in comments_flattened)

SHORT_COMMENT_LENGTH = 8
short_comments = [comment for comment in comments_flattened if len(comment['comment_text']) < SHORT_COMMENT_LENGTH]
comment_lengths = [len(comment['comment_text']) for comment in comments_flattened]

comments_with_urls = [c for c in comments_flattened if URL_PATTERN.search(c['comment_text'])]
comments_with_timestamps = [c for c in comments_flattened if TIMESTAMP_PATTERN.search(c['comment_text'])]
comments_with_mentions = [c for c in comments_flattened if MENTION_PATTERN.search(c['comment_text'])]
comments_with_digits = [c for c in comments_flattened if DIGIT_PATTERN.search(c['comment_text'])]
comments_with_punctuation = [c for c in comments_flattened if PUNCT_PATTERN.search(c['comment_text'])]

comments_with_urls_pct = 100 * len(comments_with_urls) / len(comments_flattened)
comments_with_timestamps_pct = 100 * len(comments_with_timestamps) / len(comments_flattened)
comments_with_mentions_pct = 100 * len(comments_with_mentions) / len(comments_flattened)
comments_with_digits_pct = 100 * len(comments_with_digits) / len(comments_flattened)
comments_with_punctuation_pct = 100 * len(comments_with_punctuation) / len(comments_flattened)

# For author, video, and channel distributions
video_counter = Counter(comment['video_id'] for comment in comments_flattened)
channel_counter = Counter(comment['channel_id'] for comment in comments_flattened)
author_counter = Counter(comment['comment_author'] for comment in comments_flattened)

# Print all details at bottom
print("BASIC DATA EXPLORATION")
print("=" * 80)
print(f"Total comments: {len(comments_flattened)}")
print(f"Unique videos: {len(unique_videos)}")
print(f"Unique channels: {len(unique_channels)}")
print(f"Unique authors: {len(unique_authors)}\n")

print(f"Short comments (<{SHORT_COMMENT_LENGTH} chars): {len(short_comments)}")
print(f"Comments w/ URLs: {len(comments_with_urls)} ({comments_with_urls_pct:.2f}%)")
print(f"Comments w/ timestamps: {len(comments_with_timestamps)} ({comments_with_timestamps_pct:.2f}%)")
print(f"Comments w/ mentions: {len(comments_with_mentions)} ({comments_with_mentions_pct:.2f}%)")
print(f"Comments w/ digits: {len(comments_with_digits)} ({comments_with_digits_pct:.2f}%)")
print(f"Comments w/ punctuation: {len(comments_with_punctuation)}\n")

print(f"Max comment length: {max(comment_lengths) if comment_lengths else 0}")
print(f"Average comment length: {sum(comment_lengths)/len(comment_lengths):.2f}" if comment_lengths else "Avg. comment length: 0")

BASIC DATA EXPLORATION
Total comments: 46808
Unique videos: 109
Unique channels: 73
Unique authors: 35138

Short comments (<8 chars): 229
Comments w/ URLs: 15 (0.03%)
Comments w/ timestamps: 1462 (3.12%)
Comments w/ mentions: 3730 (7.97%)
Comments w/ digits: 6880 (14.70%)
Comments w/ punctuation: 40049

Max comment length: 9817
Average comment length: 97.74


In [16]:
with open(MET_GALA_ENTITIES_PATH, "r", encoding="utf-8") as f:
    met_gala_entities = json.load(f)

total_entities = len(met_gala_entities['entities'])
celebs = [entity for entity in met_gala_entities['entities'].values() if entity['type'] == 'celebrity']
brands = [entity for entity in met_gala_entities['entities'].values() if entity['type'] == 'designer_brand']

print(f"Total entities: {total_entities}")
print(f"Total celebrities: {len(celebs)}")
print(f"Total brands: {len(brands)}")

Total entities: 481
Total celebrities: 361
Total brands: 120


In [17]:
# Prepare exact entity aliases after light cleaning is available
def build_entity_aliases(entities):
    entity_aliases = []
    for entity in entities:
        aliases = set()
        for alias in entity.get("aliases", []) + [entity["name"]]:
            clean_alias = clean_for_entity_matching(alias)
            if clean_alias:
                aliases.add(clean_alias)
        entity_aliases.append({"name": entity["name"], "aliases": sorted(aliases)})
    return entity_aliases

def find_entities(text, entity_aliases):
    clean_text = clean_for_entity_matching(text)
    padded_text = f" {clean_text} "
    found = []
    for entity in entity_aliases:
        for alias in entity["aliases"]:
            if f" {alias} " in padded_text:
                found.append(entity["name"])
                break
    return found

# Create exact alias matching for celeb and brand names
celeb_aliases = build_entity_aliases(celebs)
brand_aliases = build_entity_aliases(brands)


In [18]:
# Match celebrity and brand aliases in each comment
matched_comments = []
celeb_counter = Counter()
brand_counter = Counter()

for comment in comments_flattened:
    text_for_matching = comment.get("comment_text_entity", "")

    # Find entities in comment
    found_celebs = find_entities(text_for_matching, celeb_aliases)
    found_brands = find_entities(text_for_matching, brand_aliases)

    # Scoring on each counter
    for celeb in found_celebs:
        celeb_counter[celeb] += 1
    for brand in found_brands:
        brand_counter[brand] += 1

    comment["celebs"] = found_celebs
    comment["brands"] = found_brands

    # Append to matched comments
    matched_comments.append({
        **comment,
        "comment_id": comment.get("comment_id"),
        "video_id": comment.get("video_id"),
        "video_title": comment.get("video_title"),
        "comment_text": comment.get("comment_text", ""),
        "comment_text_entity": text_for_matching,
        "celebs": found_celebs,
        "brands": found_brands,
    })


In [19]:
comments_with_celebs = [row for row in matched_comments if row["celebs"]]
comments_with_brands = [row for row in matched_comments if row["brands"]]
# Which comments have both brand and celeb mentions 
# Expect this to be smaller
comments_with_both = [row for row in matched_comments if row["celebs"] and row["brands"]]

comments_with_celebs_pct = 100 * len(comments_with_celebs) / len(comments_flattened)
comments_with_brands_pct = 100 * len(comments_with_brands) / len(comments_flattened)
comments_with_both_pct = 100 * len(comments_with_both) / len(comments_flattened)

print("ENTITY MATCH COVERAGE")
print("=" * 80)
print(f"Total comments: {len(comments_flattened)}")
print(f"Comments with celebrity mentions: {len(comments_with_celebs)} ({comments_with_celebs_pct:.2f}%)")
print(f"Comments with brand mentions: {len(comments_with_brands)} ({comments_with_brands_pct:.2f}%)")
print(f"Comments with both celebrity and brand mentions: {len(comments_with_both)} ({comments_with_both_pct:.2f}%)")


ENTITY MATCH COVERAGE
Total comments: 46808
Comments with celebrity mentions: 10279 (21.96%)
Comments with brand mentions: 1004 (2.14%)
Comments with both celebrity and brand mentions: 349 (0.75%)


In [20]:
# Entities that are being found often enough to work with
matched_celeb_count = len(celeb_counter)
matched_brand_count = len(brand_counter)
matched_celeb_count_pct = 100 * matched_celeb_count / len(celebs)
matched_brand_count_pct = 100 * matched_brand_count / len(brands)

print("ENTITY FREQUENCY CHECK")
print("=" * 80)
print(f"Matched celebrities: {matched_celeb_count}/{len(celebs)} ({matched_celeb_count_pct:.2f}%)")
print(f"Matched brands: {matched_brand_count}/{len(brands)} ({matched_brand_count_pct:.2f}%)")


ENTITY FREQUENCY CHECK
Matched celebrities: 215/361 (59.56%)
Matched brands: 66/120 (55.00%)


In [21]:

print("\nTOP 20 CELEBRITIES")
print("=" * 80)
for celeb, count in celeb_counter.most_common(20):
    print(f"{celeb}: {count}")



TOP 20 CELEBRITIES
Beyonce: 1335
Jisoo: 1284
LISA: 1044
Rose: 935
Rihanna: 739
Madonna: 432
JENNIE: 411
Emma Chamberlain: 372
Heidi Klum: 338
Cardi B: 325
Anne Hathaway: 291
Kylie Jenner: 238
Katy Perry: 229
Bad Bunny: 223
Blake Lively: 210
Karan Johar: 202
Sabrina Carpenter: 200
Sam Smith: 183
Tyla: 181
Ningning: 169


In [22]:

print("\nTOP 20 BRANDS")
print("=" * 80)
for brand, count in brand_counter.most_common(20):
    print(f"{brand}: {count}")


TOP 20 BRANDS
Saint Laurent: 256
Robert Wun: 161
Dior: 87
Mugler: 77
Chanel: 71
Hugo Boss: 51
Balenciaga: 44
Schiaparelli: 40
Prada: 32
Zara: 24
Skims: 20
Zac Posen: 17
Gap Studio: 16
Valentino: 15
Chloe: 12
Tom Ford: 10
Christian Siriano: 9
Michael Kors: 9
Vivienne Westwood: 9
Bulgari: 9


In [23]:
# Store preprocessing variants on each filtered English comment
for comment in comments_flattened:
    original_text = comment.get("comment_text", "")
    topic_tokens = clean_for_topic_tokens(original_text)
    topic_stemmed_tokens = clean_for_topic_stemmed_tokens(original_text)

    # Used for VADER and BERT sentiment analysis
    comment["comment_text_sentiment_analysis"] = sentiment_analysis_clean(original_text)

    # Used for topic modelling
    comment["comment_text_topic"] = " ".join(topic_tokens)
    comment["comment_tokens_topic"] = topic_tokens
    comment["comment_text_topic_stemmed"] = " ".join(topic_stemmed_tokens)
    comment["comment_tokens_topic_stemmed"] = topic_stemmed_tokens
    comment["topic_token_count"] = len(topic_tokens)

print(f"Stored preprocessing variants for {len(comments_flattened)} English comments")


Stored preprocessing variants for 46808 English comments


In [ ]:
pre_language_filter_count = len(language_results)
post_language_filter_count = len(comments_flattened)
language_removed_count = pre_language_filter_count - post_language_filter_count

raw_texts = [comment.get("comment_text", "") for comment in comments_flattened]
sentiment_texts = [comment.get("comment_text_sentiment_analysis", "") for comment in comments_flattened]
topic_texts = [comment.get("comment_text_topic", "") for comment in comments_flattened]
topic_token_counts = [comment.get("topic_token_count", 0) for comment in comments_flattened]

# Row counts
row_percent_retained = 100 * post_language_filter_count / pre_language_filter_count if pre_language_filter_count else 0
row_percent_removed = 100 * language_removed_count / pre_language_filter_count if pre_language_filter_count else 0
url_removed_count = sum(1 for text in raw_texts if URL_PATTERN.search(text))
timestamp_removed_count = sum(1 for text in raw_texts if TIMESTAMP_PATTERN.search(text))
mention_removed_count = sum(1 for text in raw_texts if MENTION_PATTERN.search(text))
url_removed_pct = 100 * url_removed_count / post_language_filter_count if post_language_filter_count else 0
timestamp_removed_pct = 100 * timestamp_removed_count / post_language_filter_count if post_language_filter_count else 0
mention_removed_pct = 100 * mention_removed_count / post_language_filter_count if post_language_filter_count else 0

raw_char_count = sum(len(text) for text in raw_texts)
sentiment_char_count = sum(len(text) for text in sentiment_texts)
topic_char_count = sum(len(text) for text in topic_texts)
raw_token_count = sum(len(tokenize(text)) for text in raw_texts)
topic_token_count = sum(topic_token_counts)

# Text reduction from preprocessing
char_sentiment_reduction = 100 * (raw_char_count - sentiment_char_count) / raw_char_count if raw_char_count else 0
char_topic_reduction = 100 * (raw_char_count - topic_char_count) / raw_char_count if raw_char_count else 0
token_topic_reduction = 100 * (raw_token_count - topic_token_count) / raw_token_count if raw_token_count else 0

sentiment_changed_count = sum(1 for raw, clean in zip(raw_texts, sentiment_texts) if normalise_whitespace(raw) != clean)
sentiment_changed_pct = 100 * sentiment_changed_count / post_language_filter_count if post_language_filter_count else 0
empty_sentiment_text_count = sum(1 for text in sentiment_texts if not text.strip())
empty_sentiment_text_pct = 100 * empty_sentiment_text_count / post_language_filter_count if post_language_filter_count else 0
empty_topic_text_count = sum(1 for text in topic_texts if not text.strip())
empty_topic_text_pct = 100 * empty_topic_text_count / post_language_filter_count if post_language_filter_count else 0


PREPROCESSING STATS SUMMARY
Total rows before filter: 63,048
Rows kept (English): 46,808 (74.2%)
Rows removed (non-English): 16,240 (25.8%)

Noise removal stats:
Rows with URLs: 15 (0.03%)
Rows with timestamps: 1,462 (3.12%)
Rows with mentions: 3,730 (7.97%)
Rows changed by cleaning: 5,180 (11.1%)

Preprocessing text reduction:
Chars (sentiment): 4,575,121 -> 4,493,271 (1.8%)
Chars (topic): 4,575,121 -> 2,826,266 (38.2%)
Tokens (topic): 971,761 -> 436,276 (55.1%)

Empty text after preprocessing:
Empty sentiment text: 19 (0.04%)
Empty topic text: 283 (0.60%)


In [ ]:

print("PREPROCESSING STATS SUMMARY")
print("=" * 80)
print(f"Total rows before filter: {pre_language_filter_count:,}")
print(f"Rows kept (English): {post_language_filter_count:,} ({row_percent_retained:.1f}%)")
print(f"Rows removed (non-English): {language_removed_count:,} ({row_percent_removed:.1f}%)")
print()
print("Noise removal stats:")
print(f"Rows with URLs: {url_removed_count:,} ({url_removed_pct:.2f}%)")
print(f"Rows with timestamps: {timestamp_removed_count:,} ({timestamp_removed_pct:.2f}%)")
print(f"Rows with mentions: {mention_removed_count:,} ({mention_removed_pct:.2f}%)")
print(f"Rows changed by cleaning: {sentiment_changed_count:,} ({sentiment_changed_pct:.1f}%)")
print()
print("Preprocessing text reduction:")
print(f"Chars (sentiment): {raw_char_count:,} -> {sentiment_char_count:,} ({char_sentiment_reduction:.1f}%)")
print(f"Chars (topic): {raw_char_count:,} -> {topic_char_count:,} ({char_topic_reduction:.1f}%)")
print(f"Tokens (topic): {raw_token_count:,} -> {topic_token_count:,} ({token_topic_reduction:.1f}%)")
print()
print("Empty text after preprocessing:")
print(f"Empty sentiment text: {empty_sentiment_text_count:,} ({empty_sentiment_text_pct:.2f}%)")
print(f"Empty topic text: {empty_topic_text_count:,} ({empty_topic_text_pct:.2f}%)")


PREPROCESSING STATS SUMMARY
Total rows before filter: 63,048
Rows kept (English): 46,808 (74.2%)
Rows removed (non-English): 16,240 (25.8%)

Noise removal stats:
Rows with URLs: 15 (0.03%)
Rows with timestamps: 1,462 (3.12%)
Rows with mentions: 3,730 (7.97%)
Rows changed by cleaning: 5,180 (11.1%)

Preprocessing text reduction:
Chars (sentiment): 4,575,121 -> 4,493,271 (1.8%)
Chars (topic): 4,575,121 -> 2,826,266 (38.2%)
Tokens (topic): 971,761 -> 436,276 (55.1%)

Empty text after preprocessing:
Empty sentiment text: 19 (0.04%)
Empty topic text: 283 (0.60%)


In [25]:
# Demonstration of processing for random sample and report
RANDOM_TOTAL_EXAMPLES = 10
purify_random_samples = random.sample(english_comments, RANDOM_TOTAL_EXAMPLES)
for idx, text in enumerate(purify_random_samples):
    history = purify_text(text['comment_text'], True).items()
    print(f"\n[{idx+1}/{RANDOM_TOTAL_EXAMPLES}] {text['comment_text'][:100]}")
    for step, value in history:
        print(f"  [{step}] {value}")


[1/10] Making this a Bezos backed event when a couple of years ago Lauren Sanchez couldn't even get on a Vo
  [remove_html_entities] Making this a Bezos backed event when a couple of years ago Lauren Sanchez couldn't even get on a Vogue cover is honestly a sad way fashion (art in clothing) has sold out. 

But I guess, ever since high fashion was exposed as being sweatshop swap meet hawkers of their own designs by Chinese manufacturers and overly priced, that "Why not let the Bezos run the party since the cheap fake cat is out of the bag? We don't need to be high fashion. We just need to have a party and be gaudi while doing so."

So I guess it's the perfect Met Gala? Sign of the times and of the *cough...cough* in the .....house and ....itol.

😂😂😂
  [remove_urls] Making this a Bezos backed event when a couple of years ago Lauren Sanchez couldn't even get on a Vogue cover is honestly a sad way fashion (art in clothing) has sold out. 

But I guess, ever since high fashion was exposed as

In [26]:

processed_video_data = {
    "comments": comments_flattened,
    "entity_counts": {
        "celebs": dict(celeb_counter),
        "brands": dict(brand_counter),
    },
}

with open(PROCESSED_VIDEO_DATA_PATH, "w", encoding="utf-8") as f:
    json.dump(processed_video_data, f, ensure_ascii=False, indent=2)

print(f"Saved processed data to {PROCESSED_VIDEO_DATA_PATH}")


Saved processed data to /Users/cooper/Documents/GitHub/Network-Analysis-Project#/data/video_data_processed.json


>